In [ ]:
!pip install numpy pandas matplotlib scipy yfinance arch statsmodels -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Chart style
BLUE = '#1A3A6E'; RED = '#DC3545'; GREEN = '#2E7D32'
ORANGE = '#E67E22'; GRAY = '#666666'; PURPLE = '#8E44AD'

plt.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none',
    'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.grid': False, 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'xtick.labelsize': 9,
    'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.0, 'axes.linewidth': 0.6,
    'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.edgecolor': 'none',
    'figure.dpi': 150,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved: {name}')

def legend_bottom(ax, ncol=2, y=-0.18):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

In [ ]:
import yfinance as yf
from arch import arch_model

sp = yf.download('^GSPC', start='2000-01-01', end='2025-12-31', progress=False)
if isinstance(sp.columns, pd.MultiIndex):
    sp.columns = sp.columns.get_level_values(0)
returns = (sp['Close'].pct_change() * 100).dropna()
returns = pd.Series(returns.values, index=returns.index, name='returns')

In [ ]:
# Fit EGARCH to capture leverage
m_e = arch_model(returns, vol='EGARCH', p=1, q=1, dist='t')
r_e = m_e.fit(disp='off')
print(r_e.summary().tables[1])

gamma = r_e.params['gamma[1]']
print(f"\nLeverage parameter γ = {gamma:.4f}")
print(f"γ < 0 confirms leverage effect: negative shocks increase volatility more")

In [ ]:
# Scatter: lagged returns vs squared returns (volatility proxy)
lag_ret = returns.values[:-1]
sq_ret = returns.values[1:]**2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter plot
ax = axes[0]
ax.scatter(lag_ret, sq_ret, s=1, alpha=0.3, color=BLUE)
# Binned averages
bins = np.percentile(lag_ret, np.arange(0, 101, 5))
bin_centers = 0.5*(bins[:-1] + bins[1:])
bin_means = [np.mean(sq_ret[(lag_ret >= bins[i]) & (lag_ret < bins[i+1])]) for i in range(len(bins)-1)]
ax.plot(bin_centers, bin_means, 'o-', color=RED, lw=2, ms=4, label='Binned mean')
ax.set_xlabel('Return at t-1 (%)'); ax.set_ylabel('Squared return at t (%²)')
ax.set_title('Leverage Effect', fontweight='bold')
ax.legend(frameon=False)

# Right: asymmetric response
ax = axes[1]
neg_mask = lag_ret < 0
pos_mask = lag_ret >= 0
neg_vol = np.mean(sq_ret[neg_mask])
pos_vol = np.mean(sq_ret[pos_mask])
bars = ax.bar(['After negative\nreturns', 'After positive\nreturns'],
              [neg_vol, pos_vol], color=[RED, GREEN], width=0.5, alpha=0.8)
ax.set_ylabel('Mean squared return (%²)')
ax.set_title('Asymmetric Volatility Response', fontweight='bold')
for bar, val in zip(bars, [neg_vol, pos_vol]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
save_chart(fig, 'garch_leverage_effect')
plt.show()

print(f"\nMean vol after negative returns: {neg_vol:.2f}")
print(f"Mean vol after positive returns:  {pos_vol:.2f}")
print(f"Ratio: {neg_vol/pos_vol:.2f}x")